# What is an AI Agent

> Large language models are already capable. They write code, translate, and answer factual questions, and the text they produce reads fluently. A single call, however, still returns only a piece of **text**. The model does not touch the outside world, so it cannot carry a multi-step task from start to finish.
>
> This lecture is the starting point of the course. We begin with a concrete task that a single call cannot complete; it has to be placed in a **loop**. From that task we give **Agent** a definition used throughout the course. We then build the first minimal loop from scratch.

We start from a concrete task.

The task is this: strip debug print statements from a project, then run the tests and confirm that nothing broke. This is a task that has to actually "do" something.

A single question-and-answer exchange cannot finish it. The model itself does not call any external program. Asked to "strip the debug prints," it only outputs text describing how to edit; the real edits have to be carried out by an external script. To let the model complete the task, we have to place it in a **loop**. We draw that loop step by step:

1. Give the task to the model. From the current situation, the model decides what to do next.
2. Execute that step. For example, edit a file or run a command.
3. Feed the result of the execution back to the model.
4. Return to step 1. Repeat until the task is done.

The structure that carries this loop is the subject of this lecture. We call it an **Agent**.

Before we spell the loop out, we pin down three words that appear throughout it.

The `environment` is the world outside the Agent. The file system, the web, a database, a code executor: all of these count.

`State` is all information about the task so far: both the actions already taken and the intermediate results already obtained.

An `action` is one step that the model outputs and that can actually run in the environment: for example, editing a file or running a command.

Now we map the three words onto the opening task. The environment is the project's files and test commands. The state is which files have already been cleaned and whether the tests still pass. The actions are editing a file and running the tests.

By the end of this lecture we will have written such a loop from scratch. Once it is written, the difference between an Agent and an ordinary LLM application will be visible.

## 1. From LLM applications to Agents

The previous section drew a loop for an Agent. This section looks at the other path: the path without a loop, a single call. To see why the loop is necessary, we first look at what a single call lacks.

Send a sentence to the model; the model returns a piece of text. That round trip is one forward pass inside the model. It sounds sufficient, but using it to "do" things runs into hard limits. We first build a shared model client, make one single call, and see how the model answers a task that requires "doing" something.

In [ ]:
# Add the repository root to the module search path so we can import llm_client
import sys
import os
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, "llm_client.py")):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)

from llm_client import get_llm

# force_scripted=True keeps the demo reproducible offline; drop that argument after a real key is configured
client = get_llm()
{"client": type(client).__name__, "scripted": False}


In [ ]:
# The core action of an LLM application: put the task in a prompt, call once, get text
task = "Strip debug print statements from the project, then run the tests"
reply = client.chat([{"role": "user", "content": task}])
# Keep a short preview of the behavior; do not treat the full model reply as an experimental result.
{"reply_preview": reply[:160], "reply_chars": len(reply),
 "tool_calls": 0}

A single call has four hard limits. We take them one by one, each with concrete numbers and examples.

**Can speak, cannot act**. The model's entire output is tokens, that is, text. It cannot actually open a file, send a request, or execute code. Those operations can only be performed by a program we write. Asked to "strip the debug prints," it only outputs text describing how to edit. The edits themselves are executed by an external script.

**The context window is finite**. Both the model's input and its output are first cut into small pieces. Each piece is called a `token`. One Chinese character is roughly one token. One common English word is roughly one token. The maximum number of tokens a single call can hold is the `context window`. The input prompt and the output both count; their sum cannot exceed it. GPT-2's window is 1024 tokens, about a thousand words. Mainstream models today sit between 128K and 1M.

Converting a task into tokens shows how quickly the window fills. One line of code averages about 20 tokens. A 3000-line repository is about 60,000 tokens. That is already close to half of a 128K window, before any model output. The longer the task and the more intermediate results, the easier it is to overflow the window. That is why a long task has to be split into multiple steps and fed to the model in batches.

**Errors are not self-corrected**. In a single call the model generates an answer once. There is no execution feedback in between. It may write buggy code and never know whether the code runs, because nobody ran it. Correction needs feedback. We run the code, feed the error message back, and let the model rewrite on the next round.

**Knowledge is static**. The model saw a fixed corpus at training time. Facts after the data cutoff are unknown to it. If a library changed its API in 2025 and the training data ends in 2024, the calling convention it produces is outdated. New facts have to be retrieved externally and fed back. That is exactly the work tools do inside the Agent loop.

The four limits fall into two groups.

- The first two are structural: output cannot execute; the window cannot hold everything.
- The last two are informational: no feedback; knowledge is stale.

Every stage of the Agent loop patches one of these. Actions are executed by tools, feedback is brought back by the loop, and new knowledge is obtained by external retrieval.

The four limits say that a single call is not enough. The table below compares it with an Agent, item by item.

| Dimension | Pure LLM application | LLM-based Agent |
|:---|:---|:---|
| Calling pattern | Single (or fixed-round) Q&A | Loop until a stopping condition |
| State | None (or session history only) | Explicit task state and memory |
| Action | Tokens only | Executable tool calls |
| Environment feedback | None | Yes (results return to context) |
| Goal | Produce an answer | Complete a task (may fail, may retry) |

State and action were illustrated with the cleanup task at the start. Here we add one term not yet defined: a `tool call` is a concrete form of action. The model outputs a structured string such as search("CS329A agents"). A program parses that string and calls the corresponding function. Whether the action can run depends on whether the matching tool was prepared in the loop.

One easy confusion: automatically assembling a prompt in code and calling an API once is using the LLM as a function call. Without a loop, state, and actions, it is still an application, not an Agent. The test is that the loop, the state, and the actions are all required.

## 2. Definition and components of an Agent

The previous section described an Agent as a loop. This section discusses more formally what an Agent is. We look at three things: where the term comes from, what the loop is made of, and what each piece looks like in code.

The classic definition of an agent predates large language models. Wooldridge's 1995 survey puts it this way: an agent is an entity situated in some environment. It perceives the environment through sensors and acts on the environment through actuators. Two points in this definition recur throughout later discussion.

1. `autonomy`. Without direct external intervention, the agent decides its own actions in order to reach a goal. In software, autonomy means the Agent chooses the actions. Given the goal "tidy the repository README," it decides which files to read first and in what order to edit, rather than having every step specified by a human.
2. `situatedness`. It faces a partially observable, continually changing environment, not a well-formed input. Partially observable means that at any moment the Agent can see only part of the environment. A program that has scanned a file list does not know which debug prints remain inside a given file. "Well-formed input" refers to ordinary programs: the input structure is fixed, the fields are complete, the program computes from start to finish and does not need to look at the outside world along the way. An Agent faces incomplete, changing information, so it has to perceive and act repeatedly.

Sensors and actuators in the definition have concrete counterparts in software.

- Sensors are the Agent's entry point for environmental information. For a person they are eyes and ears. For a software Agent they are reading files, retrieving web pages, querying a database.
- Actuators are the Agent's means of changing the environment. For a person they are hands and feet. For a software Agent they are editing files, sending requests, running commands.

An LLM-based Agent is the modern version. It uses an LLM as the decision core and calls the LLM inside a loop. It perceives, reasons, and decides, then turns the decision into an action in the environment, usually through tools, then feeds the result back as input, until the goal is reached. The LLM plays the role of the brain, not the whole system. The loop's minimal composition has five parts. We list them in a short piece of code first.

In [ ]:
# Minimal composition of the Agent loop, listed together
components = [
    ("Perception", "Assemble environment state and tool returns into context"),
    ("Decision", "The LLM outputs the next action from state and goal"),
    ("Action", "Parse the action and execute it on a tool"),
    ("Feedback", "The result returns to context as the next round of perception"),
    ("Termination", "The model gives a Final Answer, or the step budget is exhausted"),
]
for name, desc in components:
    print(f"{name} — {desc}")


The body of the loop is the five parts listed above. Before we implement it, a few more pieces need a brief account. They are often taught as independent modules; they are accessories of the loop. The three accessories are memory, planning, and verification. We look at each once.

`Memory` brings back information that sits outside the context window. Short-term memory is the message history: the list of all messages in the current conversation. Long-term memory is external storage, such as a vector store or a file. Write the previous session's conclusions into a file and read them back in the next session.

`Planning` decomposes a goal into subgoals, or searches over sequences of actions. Decomposition: the task "write course notes" is split into "look up the outline, retrieve sources, write the body, proofread," then executed step by step. Search tries several action routes at once and keeps the one that reaches the goal.

`Verification` adds a check outside the loop, so that output is accepted only after it passes. Code the model generates is run through tests first; only if the tests pass does it become the final answer.

They are not the body of the loop, but each attaches to some stage of it. We return to them in the lectures that cover them.

They are called accessories because the loop still runs without them. A loop with only an LLM, tools, and a message history can finish a small task. The minimal loop we are about to implement is of that kind.

The body of the loop is four steps plus a stopping condition. We implement it from scratch.

## 3. An Agent loop

Writing the composition from the previous section as runnable code is the first hands-on goal of this lecture. The loop body does four things.

1. Call the LLM and obtain a reply.
2. Parse the action in the reply.
3. Execute the action and write the observation back into the message history.
4. Repeat the three steps above until the model gives a `Final Answer` or the step budget is exhausted.

The loop remembers where it is through the `message history`. The message history is a list of messages maintained inside the loop. Each message records a role and a piece of content, of the form {"role": "user", "content": "..."}. The first message at the start is the task itself. After that, each round, the Agent's output is appended with the assistant role. Tool results are appended with the user role, because to the model a tool result is a new utterance in the conversation. Each round the model sees only this list. What it said last round and what the tool returned all depend on records in the list. The longer the list, the more context the model can use. So the message history is the state of the loop.

The task is a multi-step arithmetic problem: compute (3+5)×(7-2). We first compute it by hand, step by step, to give the code a reference.

In [ ]:
# Hand calculation: without tools, compute (3+5)×(7-2) step by step
step1 = 3 + 5
step2 = 7 - 2
step3 = step1 * step2
print(f"Step 1 (3+5) = {step1}")
print(f"Step 2 (7-2) = {step2}")
print(f"Step 3 8×5   = {step3}")
print("The hand calculation should match the Agent loop later")


The code above already computed the result. Next we write the same process as a message history, and watch what is added to the list at each step. The task is fixed as "Please compute (3+5)×(7-2) step by step."

At step 1, the message list contains only the task:

```text
[User] Please compute (3+5)×(7-2) step by step
```

At step 2, the model outputs Action: calc("3+5"). That line is appended with the assistant role. After the tool runs, the result is written back with the user role. Steps 3 and 4 repeat the same process. The model first outputs Action: calc("7-2") and gets 5, then Action: calc("8×5") and gets 40. At the end of step 4, the full list is:

```text
[User] Please compute (3+5)×(7-2) step by step
[Agent] Action: calc("3+5")
[User] Observation: 8
[Agent] Action: calc("7-2")
[User] Observation: 5
[Agent] Action: calc("8×5")
[User] Observation: 40
```

The list the model reads at each step has two more messages than at the previous step. At the last step, the model sees 8, 5, and 40 in the list. It outputs Final Answer: 40. That 40 is read from the list, not computed in the model's head. Tool results return to the model through the message list. That is how state flows in the loop.

After the loop receives a model reply, the first job is not to execute, but to read the text. We have to decide: the model wants to call a tool, or it is announcing completion. The component that does this is the parser. It turns text into data the loop can read directly.

A model reply can contain only three kinds of content: `Thought` plus `Action`, `Final Answer`, or ordinary text. The parser recognizes these three cases and returns a uniform (kind, action, payload) structure.

One example of each.

**1. Reasoning plus action**. The model thinks and names the tool to call:

```text
Thought: We need to retrieve the course materials first.
Action: search("CS329A agents")
```

After Action come the tool name and arguments. The parser extracts that line into a structured (tool name, arguments) pair so the loop knows which function to call.

**2. Final answer**, meaning the task is done:

```text
Final Answer: CS329A is Stanford's AI Agent course
```

**3. Ordinary text**. There is neither Action nor Final Answer. For example, the model restates the task. The parser returns "no action" for this kind of text, and the loop does not call a tool.

The order of parsing matters. Look for Final Answer first, then Action. The model may give both an Action and a Final Answer in the same reply. In that case the loop should finish executing the action, then stop on the final answer. The three fields in (kind, action, payload) answer three questions: whether to stop, which tool to run, and what the final answer is.

In [ ]:
import re


def parse_response(text):
    """Parse a model reply into a structured action.

    Returns (kind, action, payload):
    - kind: "final" / "action" / "idle"
    - action: (tool name, arguments) or None
    - payload: the final answer (when kind is final), otherwise None
    When a reply contains both Action and Final Answer, kind is final,
    but action is still returned, so the loop can execute first and then stop.
    """
    final = re.search(r"Final Answer:\s*(.+)", text, re.DOTALL)
    action = re.search(r"Action:\s*(\w+)\s*\((.*?)\)", text, re.DOTALL)
    action_pair = (action.group(1), action.group(2).strip()) if action else None
    if final:
        return ("final", action_pair, final.group(1).strip())
    if action:
        return ("action", action_pair, None)
    return ("idle", None, None)


# Test the parser on a few texts: Action only, Final only, ordinary text, both at once
samples = [
    'Thought: Search first.\nAction: search("CS329A agents")',
    "Thought: Retrieval is done.\nFinal Answer: This is the conclusion",
    "Simulated reply: this is ordinary text",
    'Thought: There is an action and a conclusion.\nAction: search("x")\nFinal Answer: conclusion',
]
for text in samples:
    kind, action, payload = parse_response(text)
    print(f"kind={kind!r}  action={action}  payload={payload!r}")


In [ ]:
class AgentLoop:
    """Minimal Agent loop: perceive → decide → execute → feedback, until stop.

    Parameters:
    - brain: callable that receives the message history and returns model text
    - tools: dict mapping tool names to callables
    """

    def __init__(self, brain, tools):
        self.brain = brain
        self.tools = tools
        self.messages = []

    def execute_tool(self, name, args):
        """Look up the tool in the registry and run it; return a result string."""
        if name not in self.tools:
            return f"Error: unknown tool {name}"
        clean_args = args.strip().strip("\"'")
        return str(self.tools[name](clean_args))

    def step(self):
        """Run one round: call the brain, parse the action, run the tool. Return (stopped, executed tool names)."""
        reply = self.brain(self.messages)
        self.messages.append({"role": "assistant", "content": reply})
        kind, action, _ = parse_response(reply)
        executed = []
        if action is not None:
            name, args = action
            result = self.execute_tool(name, args)
            self.messages.append({"role": "user", "content": f"Observation: {result}"})
            executed.append(name)
        has_final = kind == "final"
        return has_final, executed

    def run(self, task, max_steps=10):
        """Run the loop until the model gives a Final Answer or the step budget is exhausted; return the full message history."""
        self.messages = [{"role": "user", "content": task}]
        for _ in range(max_steps):
            has_final, _ = self.step()
            if has_final:
                break
        else:
            self.messages.append({"role": "user",
                                  "content": "Error: step budget exhausted, no Final Answer"})
        return self.messages


In [ ]:
def calc(expr):
    """Evaluate a one-step arithmetic expression; supports one addition, subtraction, or multiplication."""
    expr = expr.strip()
    for op, fn in [("+", lambda a, b: a + b),
                   ("-", lambda a, b: a - b),
                   ("×", lambda a, b: a * b)]:
        if op in expr:
            left, right = expr.split(op, 1)
            return fn(int(left), int(right))
    raise ValueError(f"Cannot parse expression: {expr}")


def arithmetic_brain(messages):
    """Scripted arithmetic brain: call calc on a fixed plan, reading intermediate results from observations.

    In a live run this step is done by the llm_client model; here a fixed trajectory keeps the demo offline.
    """
    issued = " ".join(m["content"] for m in messages)
    plan = ['calc("3+5")', 'calc("7-2")', 'calc("8×5")']
    for action in plan:
        if action not in issued:
            return f"Thought: Intermediate state has been written back; continue the plan.\nAction: {action}"
    last_obs = [m for m in messages if m["content"].startswith("Observation")]
    final = last_obs[-1]["content"].split(": ", 1)[1] if last_obs else "40"
    return f"Final Answer: {final}"


tools = {"calc": calc}
task = "Please compute (3+5)×(7-2) step by step, one calc call per step, then give a Final Answer."
loop = AgentLoop(brain=arithmetic_brain, tools=tools)
trace = loop.run(task, max_steps=6)
for msg in trace:
    tag = "User" if msg["role"] == "user" else "Agent"
    print(f"[{tag}] {msg['content']}")
    print()
print("Key observation: the intermediate results 8, 5, and 40 are written back into the message history; the loop advances on that state")


One common misunderstanding needs a correction. An Agent loop is not the same as multi-turn chat.

ChatGPT's multi-turn chat only remembers turns. The model is still generating text each time. There is no action and no feedback.

Every round of an Agent loop goes through the environment, even if that is only reading a file. The test is whether there is an executable action that changes the world. In the previous section, `calc` is such an action. It really evaluates arithmetic and returns a result.

Put the two traces side by side.

- Multi-turn chat: a user line, a model line. The list has no Observation.
- Agent loop: the model outputs Action, the program runs a tool, and Observation is fed back. The list contains at least one record from a tool.

Reading a file counts as an action, because it really reads content from the environment. That content enters the model's context on the next round. Conversely, a model output with no Action is only one turn of multi-turn chat.

We now swap the loop's brain for llm_client's scripted client, and watch how the loop handles a scripted trajectory from the model.

In [ ]:
# Same loop code, brain swapped for llm_client's scripted client; behavior changes with it
search = lambda q: f"Retrieved 3 documents related to {q}"
tools = {"search": search}

task = ("Please retrieve CS329A course information and give a conclusion.\n"
        "Use the following format at each step:\n"
        "Thought: ...\n"
        "Action: search(\"keyword\")\n"
        "or give Final Answer: conclusion")

loop = AgentLoop(brain=client.chat, tools=tools)
trace = loop.run(task, max_steps=3)
for msg in trace:
    tag = "User" if msg["role"] == "user" else "Agent"
    print(f"[{tag}] {msg['content']}")
    print()
print("Key observation: the scripted client puts Action and Final Answer in the same reply; the loop executes first, then stops")


In [ ]:
# Quantify the gap between a single call and a loop with two numbers
def count_tool_calls(messages):
    """Count tool actions executed in the message history (counted as Observation messages)."""
    return sum(1 for m in messages if m["content"].startswith("Observation"))


single_context = len(reply)
loop_context = sum(len(m["content"]) for m in trace)
print(f"Single call: context {single_context} characters, tool calls 0")
print(f"Agent loop: context {loop_context} characters, tool calls {count_tool_calls(trace)}")
print("Key observation: the capability gain comes from the loop and tool feedback, not from the model itself")


## 4. A map of the Agent ecosystem

We have just implemented a loop of our own from scratch. Now we step back. Around the Agent loop, several mature approaches have formed in the field; they are called paradigms. Knowing them shows where our loop sits on the map, and how others build loops. This section lists the main paradigms.

One clarification first. These paradigms are not mutually exclusive categories. They are several answers to "where does the decision come from." Real systems are usually combinations. We list six paradigms, a representative piece of work for each, and where they sit in this course.

In [ ]:
# Quick cards for six main paradigms, plus two real-scenario selection drills
paradigms = {
    "ReAct": ("Alternate reasoning and acting, weaving thought and doing into one trajectory", "Yao et al. 2022", "L4"),
    "Tool use": ("The model outputs structured tool calls; tool results become input", "Toolformer / MCP", "L4"),
    "Planning": ("Decompose the task first, or search the action space", "LATS", "L5"),
    "Multi-agent": ("Several Agents collaborate, debate, and divide labor", "AutoGen", "L6/L7"),
    "Memory-based": ("Explicit long-term memory, across sessions and long tasks", "MemGPT", "L11"),
    "Frameworks": ("Package the loop, tools, and memory as a library", "LangGraph / Claude SDK", "referenced throughout"),
}
for name, (desc, work, pos) in paradigms.items():
    print(f"{name:<14} {desc}  |  {work}  |  {pos}")

print()
scenarios = [
    ("A coding assistant that reads several files, runs tests, and fixes bugs round by round", "ReAct + Tool use"),
    ("A long report that outlines first, then retrieves sources section by section and writes", "Planning + Memory"),
    ("A teacher and a student take turns setting and answering questions, to generate training data", "Multi-agent"),
]
for scene, answer in scenarios:
    print(f"Scenario: {scene}")
    print(f"  Suitable paradigm: {answer}")
print("Key observation: paradigms are not mutually exclusive; a real Agent is often ReAct + Planning + Memory at once")


Of the six paradigms, ReAct and Planning sit closest to the loop itself. We take these two first. The other four get one sentence each.

ReAct alternates reasoning and acting. Each round the model first outputs a Thought, explaining why it is doing this, then an Action naming the tool to call. After the environment returns an Observation, the next round begins. The Agent loop above has this shape. Thought, Action, and Observation alternate. Our parser simply does not extract Thought on its own.

Planning plans before acting. It decomposes a goal into subgoals, or searches over sequences of actions. Decomposition: "write course notes" is listed as "look up the outline, retrieve sources, write the body, proofread," then executed step by step. Search tries several action routes at once and keeps the one that reaches the goal; LATS belongs to this family. In one sentence: ReAct looks and acts as it goes; Planning plans first, then acts.

One example of each of the remaining four.

- **Tool use**: the model outputs structured tool calls. Toolformer is an early representative. Today's MCP is a standardized interface for this kind of call.
- **Multi-agent**: several Agents converse and collaborate. A teacher and a student taking turns setting and answering questions is a typical use.
- **Memory-based**: explicit long-term memory across sessions. MemGPT manages long conversations with layered memory. Content that exceeds the context is first stored externally and retrieved when needed.
- **Frameworks**: package the loop, tools, and memory as a library. LangGraph and the Claude SDK belong here. They can be referenced throughout.

Real systems rarely use only one paradigm. ReAct as the backbone, Planning to decompose long tasks, Memory to remember history, and Tool use to attach external tools: that is a common combination.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(10, 6.5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6.5)
ax.axis("off")
ax.set_title("The Agent Loop", fontsize=13)

# Four stages of the main loop (labels in the figure are in English)
nodes = [
    (5.0, 5.6, "Observation"),
    (8.5, 3.0, "Reason + Action"),
    (5.0, 0.4, "Execution"),
    (1.5, 3.0, "Feedback"),
]
for x, y, label in nodes:
    box = mpatches.FancyBboxPatch((x - 1.5, y - 0.6), 3.0, 1.2,
                                  boxstyle="round,pad=0.08",
                                  fc="#e8eefb", ec="#4a6fa5", lw=1.4)
    ax.add_patch(box)
    ax.text(x, y, label, ha="center", va="center", fontsize=11)

# Clockwise loop arrows
def arrow(p1, p2):
    ax.annotate("", xy=p2, xytext=p1,
                arrowprops=dict(arrowstyle="-|>", color="#333333", lw=1.8))

arrow((5.0, 4.9), (7.0, 3.7))   # Observation -> Reason + Action
arrow((7.1, 2.6), (6.2, 1.0))   # Reason + Action -> Execution
arrow((3.8, 1.0), (2.9, 2.6))   # Execution -> Feedback
arrow((3.0, 3.7), (3.5, 4.9))   # Feedback -> Observation

# Accessories: attached to the corresponding stage, with lecture numbers
accessories = [
    (0.5, 5.6, "Verification (L3)"),
    (9.5, 5.6, "Memory (L11)"),
    (9.5, 0.4, "Planning (L5)"),
    (0.5, 0.4, "Training (L6-L9)"),
]
for x, y, label in accessories:
    box = mpatches.FancyBboxPatch((x - 1.1, y - 0.4), 2.2, 0.8,
                                  boxstyle="round,pad=0.05",
                                  fc="#f7f2e6", ec="#a58a4a", lw=1.2)
    ax.add_patch(box)
    ax.text(x, y, label, ha="center", va="center", fontsize=9)

plt.tight_layout()
plt.show()


## 5. Course roadmap

By the end of this lecture we have seen the skeleton of the loop: perception, decision, action, feedback.

A real Agent still lacks many capabilities. This course lines up 17 notebooks in the order "fill what is missing." This section unfolds the full course map, so we can look up where any lecture sits.

The course runs in four parts, 17 notebooks in total.

- Part 1 adds capability. Test-time compute (spending extra compute at inference), verification, tools, planning. Starting from "an LLM can only generate," we add layers.
- Part 2 adds how it gets stronger. Train-time scaling, open-ended evolution, search, post-training evolution.
- Part 3 adds engineering. SWE, memory, evaluation.
- Part 4 adds the frontier. Reasoning, mathematics, autonomy, robotics.

We list the lectures in each part.

In [ ]:
# Course map: four parts, 17 notebooks
roadmap = [
    ("Part 1  Foundation",
     ["L1 What is an AI Agent", "L2 Test-time compute", "L3 Robust verification",
      "L4 Tools and code feedback", "L5 Multi-step planning"]),
    ("Part 2  Training",
     ["L6 Train-time scaling", "L7 Open-ended evolution", "L8 Search and deep research", "L9 Post-training evolution"]),
    ("Part 3  Engineering",
     ["L10 SWE Agent", "L11 Memory systems", "L14 Evaluation"]),
    ("Part 4  Frontiers",
     ["L12 LLM reasoning", "L13 Math Agent", "L15 Autonomous agents",
      "L16 Multimodal robotics", "L17 Future directions"]),
]
for part, lectures in roadmap:
    print(part, "—", ", ".join(lectures))
total = sum(len(lectures) for _, lectures in roadmap)
print(f"\nTotal: {total} notebooks")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

parts = ["Part 1: Foundation", "Part 2: Training",
         "Part 3: Engineering", "Part 4: Frontiers"]
counts = [5, 4, 3, 5]
colors = ["#4a6fa5", "#6a9fb5", "#9ab0a5", "#b58a6a"]

fig, ax = plt.subplots(figsize=(8, 4))
y = np.arange(len(parts))
ax.barh(y, counts, color=colors, edgecolor="#333333")
for i, c in enumerate(counts):
    ax.text(c + 0.08, i, str(c), va="center", fontsize=11)
ax.set_yticks(y)
ax.set_yticklabels(parts, fontsize=10)
ax.set_xlabel("Number of notebooks", fontsize=10)
ax.set_title("Course Roadmap: 4 Parts, 17 Notebooks", fontsize=12)
ax.set_xlim(0, 6.5)
plt.tight_layout()
plt.show()


Every lecture follows the same teaching contract: intuition → hand calculation → implementation → experiment. This lecture has already demonstrated the first few steps. We first built intuition for the loop, then walked through it on paper, then wrote it from scratch. Each later lecture repeats this rhythm.

By the end of this lecture we have written the first minimal loop. One `AgentLoop` class, one tool registry, one parser that can read actions. Together they are under fifty lines. That is enough to show one thing. The gap between an Agent and an application is in the structure, not in the model. The loop connects environment, tools, memory, and multi-step compute to the model. The capability gain comes from that.

The next lecture starts with test-time compute. After training is finished, we look at how far extra compute spent at inference can compensate for the quality of a single generation.

## Summary

What this lecture covered:

- [ ] An Agent is a loop: the LLM chooses the next action from the current state, the action runs in the environment, the observation is fed back to the LLM, until the task is done
- [ ] A single LLM call has four hard limits: it cannot act, context is finite, it does not self-correct, and knowledge is static
- [ ] The context window is the maximum number of tokens a single call can read; anything beyond it cannot enter the model's view
- [ ] Calling an LLM in code is not an Agent; loop, state, and action are all required
- [ ] The loop's state is carried by the message history: each round's reply and tool observation are recorded; the model sees only what is in the list
- [ ] Minimal composition of the Agent loop: perception, decision, action, feedback, termination
- [ ] Autonomy means acting without human intervention; situatedness means facing a partially observable, continually changing environment
- [ ] Memory, planning, and verification are accessories of the loop, not its body
- [ ] An Agent loop is not multi-turn chat; the test is whether there is an action that changes the world
- [ ] The six paradigms are not mutually exclusive; a real Agent is often a combination
- [ ] The course runs in four parts; each concept advances by intuition → hand calculation → implementation → experiment

## Exercises

> You may ask an AI to explain the idea. Do not ask it to finish the exercise for you.


**Exercise 1: Complete the loop's termination and write-back**

`TinyLoop.run` below has two blanks: one writes the observation back into the message history after an action runs, and one gives a readable failure message when the step budget is exhausted. The reference answers are already filled in; complete them on paper first, then run and compare. The task is fixed as "use a tool to compute 7+8, then reverse the result string."

Hint: the loop terminates in two cases — the model outputs `Final Answer`, or the step budget is exhausted; the latter message should make it clear that the task was not completed.


In [ ]:
def reverse_brain(messages):
    """Scripted brain: first call calc for 7+8, then call reverse on the result string."""
    issued = " ".join(m["content"] for m in messages)
    if "calc" not in issued:
        return 'Action: calc("7+8")'
    if "reverse" not in issued:
        return 'Action: reverse("15")'
    return "Final Answer: 51"


class TinyLoop:
    """Minimal loop: brain, tools, and message history."""

    def __init__(self, brain, tools):
        self.brain = brain
        self.tools = tools
        self.messages = []

    def execute_tool(self, name, args):
        """Look up the tool in the registry and run it; return a result string."""
        if name not in self.tools:
            return f"Error: unknown tool {name}"
        return str(self.tools[name](args.strip().strip("\"'")))

    def run(self, task, max_steps=5):
        """Run the loop; return success and the full message history."""
        self.messages = [{"role": "user", "content": task}]
        for _ in range(max_steps):
            reply = self.brain(self.messages)
            self.messages.append({"role": "assistant", "content": reply})
            kind, action, _ = parse_response(reply)
            if action is not None:
                name, args = action
                result = self.execute_tool(name, args)
                # Blank 1: write the observation back into the message history
                self.messages.append({"role": "user",
                                      "content": f"Observation: {result}"})
            if kind == "final":
                return True, self.messages
        # Blank 2: give a readable failure message when the step budget is exhausted
        return False, self.messages + [{"role": "user",
                                        "content": "Error: step budget exhausted, no final answer"}]


tools = {"calc": calc, "reverse": lambda s: s[::-1]}
tiny = TinyLoop(brain=reverse_brain, tools=tools)
ok, trace = tiny.run("Please use tools to compute 7+8, then reverse the result string", max_steps=5)

assert ok, "The loop should give a Final Answer within 5 steps"
assert trace[-1]["content"].startswith("Final Answer"), "The last message should be the final answer"
assert sum(m["content"].startswith("Observation") for m in trace) == 2, "There should be two observations"
print("Exercise 1 passed: the loop stopped within 5 steps, and intermediate results were written back into the message history")


**Exercise 2: An action parser**

Write a simpler `parse_action` from scratch, returning a `(kind, name, args)` triple: support `Action`, `Final Answer`, and no-action, and handle extra whitespace and multiple lines correctly. The reference answers are already filled in; complete them on paper first, then run and compare.

Hint: look for Final Answer before Action, because the Final line should not be treated as an Action.


In [ ]:
import re


def parse_action(text):
    """Parse a model reply into (kind, name, args).

    When kind is "final", name is None and args is the answer;
    when kind is "action", name is the tool name and args are the arguments;
    when kind is "idle", name and args are both None.
    """
    final = re.search(r"Final Answer:\s*(.+)", text, re.DOTALL)
    action = re.search(r"Action:\s*(\w+)\s*\((.*?)\)", text, re.DOTALL)
    if final:
        return ("final", None, final.group(1).strip())
    if action:
        return ("action", action.group(1), action.group(2).strip())
    return ("idle", None, None)


# Edge cases: extra whitespace, multiple lines, Final and Action in the same reply
assert parse_action('Action: add(1, 2)') == ("action", "add", "1, 2")
assert parse_action('Final Answer: 42') == ("final", None, "42")
assert parse_action('  look at  this  \n\n') == ("idle", None, None)
assert parse_action('Action:  add( 3 , 5 )') == ("action", "add", "3 , 5")
assert parse_action('Thought: think first\nAction: search(x)')[0] == "action"
assert parse_action('Final Answer: ok\nAction: foo(1)')[0] == "final"
print("Exercise 2 passed: all three cases, extra whitespace, multiple lines, and priority parse correctly")


**Exercise 3: Quantifying single-call vs loop**

On the same task, run a single call and an Agent loop. First write `count_tool_calls` yourself to count tool calls in the message history, then run and compare. Assert that the loop's tool-call count is greater than 1 and that the context contains tool return values, while the single call has neither.

Hint: tool return values are written back into context by us, so "the loop obtained tool results" is assertable and does not depend on a particular model.


In [ ]:
def count_tool_calls(messages):
    """Count tool actions executed in a message history."""
    # Blank: the number of Observation messages is the number of tool calls
    return sum(1 for m in messages if m["content"].startswith("Observation"))


# Same task: single call vs Agent loop
single_reply = client.chat([{"role": "user", "content": "Please compute (3+5)×(7-2) step by step"}])
single_trace = [{"role": "user", "content": single_reply}]

loop_tools = {"calc": calc}
loop = AgentLoop(brain=arithmetic_brain, tools=loop_tools)
loop_trace = loop.run("Please compute (3+5)×(7-2) step by step", max_steps=6)

assert count_tool_calls(loop_trace) > 1, "The loop should call tools more than once"
assert any(m["content"].startswith("Observation") for m in loop_trace), "The loop should obtain tool return values"
assert count_tool_calls(single_trace) == 0, "A single call should have no tool calls"
print(f"Single call returned: {single_reply}")
print("Exercise 3 passed: the loop's tool-call count is greater than 1; the single call is 0")


## References

- Wooldridge & Jennings, [Intelligent Agents: Theory and Practice](https://www.csc.liv.ac.uk/~mjw/pubs/ker95.pdf), 1995 — source of the classic Agent definition: situated, autonomous, perceive–act loop
- Wooldridge, [Intelligent Agents (Chapter 1 of Multiagent Systems)](https://www.cs.ox.ac.uk/people/michael.wooldridge/pubs/maia-chapter.pdf), 1999 — a fuller agent definition, including rational agents and BDI
- Yao et al., [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629), 2022 — the paradigm of alternating reasoning and acting; the default Agent-loop skeleton of this course
- Wang et al., [A Survey on LLM-based Autonomous Agents](https://arxiv.org/abs/2308.11432), 2023 — a consensus summary of Agent components: planning, memory, tool use
- Xi et al., [The Rise and Potential of Large Language Model Based Agents](https://arxiv.org/abs/2309.07864), 2023 — a broader survey of LLM-based Agents and ecosystem taxonomy
- Weng, [LLM Powered Autonomous Agents](https://lilianweng.github.io/posts/2023-06-23-agent/), 2023 — one of the clearest blog posts on the Agent loop; a good first extra reading
- Schick et al., [Toolformer: Language Models Can Teach Themselves to Use Tools](https://arxiv.org/abs/2302.04761), 2023 — an early representative of LLMs learning to call tools
- Zhou et al., [Language Agent Tree Search Unifies Reasoning, Acting, and Planning](https://arxiv.org/abs/2310.04406), 2023 — the planning/search paradigm (LATS)
- Packer et al., [MemGPT: Towards LLMs as Operating Systems](https://arxiv.org/abs/2310.08560), 2023 — the memory paradigm
- Wu et al., [AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation](https://arxiv.org/abs/2308.08155), 2023 — a multi-agent collaboration framework
- Anthropic, [Model Context Protocol](https://modelcontextprotocol.io), 2024 — a standardized interface for tools and context; the interconnect protocol of the Agent ecosystem
- Stanford, [CS329A Course Homepage](https://cs329a.stanford.edu/), 2025 — source of this course's outline and assignments
